In [ ]:
import os 
import sys
sys.path.append('../../..')

import numpy as np
import pandas as pd
# from scipy import stats
# from statsmodels.stats.multitest import multipletests
# import seaborn as sns
# import matplotlib.pyplot as plt

import scanpy as sc
import h5py


# Paths for jin-2025 dataset
PATHS = {'data_dir':  '/Users/catherinewang/Documents/School/genData/jin-2025/',
         'save_dir': '/Users/catherinewang/Documents/School/proximal-scCRISPR-github/jin-2025/saves',
         'util_dir': '/Users/catherinewang/Documents/School/proximal-scCRISPR-github/utils'}
PATHS['save_dir'] = PATHS['save_dir'] + '/1_initial'

# Load in h5ad

In [2]:
jin2025_10pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_10perturbations.h5ad')
jin2025_48pert = sc.read_h5ad(PATHS['data_dir'] + 'subset_adata_SCARF_10celltypes_48perturbations.h5ad')

In [3]:
jin2025_10pert.obs_names = jin2025_10pert.obs_names + ':10pert'
jin2025_48pert.obs_names = jin2025_48pert.obs_names + ':48pert'

# 

Concatenate the two datasets

In [4]:
jin2025 = sc.concat({'10pert': jin2025_10pert, '48pert': jin2025_48pert}, axis='obs')

Subselect genes that are most important/top or a grna target or a housekeeping gene

In [5]:
gene_dev_df =  pd.read_csv(PATHS['save_dir'] + '/gene_dev.csv')
normalized_genes_names = gene_dev_df.loc[gene_dev_df['is_normalized_gene']]['gene_name'].sort_values()

In [6]:
normalized_genes_names

10028    AI593442
4313     AU040320
5541         Aacs
10145       Aagab
6392         Aak1
           ...   
19058      mt-Nd2
19064      mt-Nd3
19066      mt-Nd4
19065     mt-Nd4l
19067      mt-Nd5
Name: gene_name, Length: 4814, dtype: str

Save as `h5ad` (special h5 format: annotated data)

In [43]:
jin2025[:, normalized_genes_names].write_h5ad(
    PATHS['save_dir'] + '/jin2025_subsetgenes.h5ad',
    compression=None
)

Save as `hdf5` (so that scry package can use)

In [59]:
gene_counts_anndata = jin2025[:, normalized_genes_names].to_memory()
gene_counts = gene_counts_anndata.X.copy()

In [60]:
with h5py.File(PATHS['save_dir'] + "/jin2025_subsetgenes.hdf5", "w") as f:
    fgroup = f.create_group('counts')

    # actual counts
    fgroup.create_dataset(name="counts", shape=gene_counts.shape, dtype='i', data = gene_counts.toarray())

    # gene names
    fgroup.create_dataset(name="col_genenames", 
                        shape=gene_counts_anndata.var_names.shape,
                        dtype=h5py.string_dtype(encoding='utf-8'), 
                        data = gene_counts_anndata.var_names)
    
    # cell names
    fgroup.create_dataset(name="row_cellnames", 
                        shape=gene_counts_anndata.obs_names.shape,
                        dtype=h5py.string_dtype(encoding='utf-8'), 
                        data = gene_counts_anndata.obs_names)

In [75]:
f =  h5py.File(PATHS['save_dir'] + "/jin2025_subsetgenes.hdf5", "r")

In [ ]:
f.keys()
gene_norm_h5 = f['gene_norm/gene_norm'] # need to keep the connection open


In [79]:
gene_norm_h5.shape

(253587, 4814)

In [78]:
gene_norm_h5[1:4, 1:4]

array([[-1.10104278, -0.13318125,  1.61490831],
       [-1.08660081,  0.74012033,  0.7457781 ],
       [-1.7045359 ,  3.49188886,  0.30119363]])

In [ ]:
with h5py.File(PATHS['save_dir'] + "/jin2025_subsetgenes.hdf5", "r") as f:
    test_gene_norm = list(f['gene_norm/gene_norm'])

In [ ]:
test_gene_norm

<Closed HDF5 dataset>

In [80]:
f.close()

In [ ]:
f = h5py.File(PATHS['save_dir'] + "/jin2025_subsetgenes.hdf5", "w")
dset = f.create_group('counts')

dset.create_dataset(name="counts", shape=gene_counts.shape, dtype='i', data = gene_counts.toarray())
dset.create_dataset(name="col_genenames", 
                    shape=gene_counts_anndata.var_names.shape,
                    dtype=h5py.string_dtype(encoding='utf-8'), 
                    data = gene_counts_anndata.var_names)
dset.create_dataset(name="row_cellnames", 
                    shape=gene_counts_anndata.obs_names.shape,
                    dtype=h5py.string_dtype(encoding='utf-8'), 
                    data = gene_counts_anndata.obs_names)

f.close()

In [ ]:

with h5py.File(PATHS['save_dir'] + "/jin2025_subsetgenes.hdf5", "w") as f:
    f.create_dataset(name="counts", shape=gene_counts.shape, dtype='i', data = gene_counts.toarray()) #, compression = 'lzf')
    

In [33]:
np.array(['hi', 'bye', 'hello'])

array(['hi', 'bye', 'hello'], dtype='<U5')

In [53]:
vec_to_save = list(gene_counts_anndata.var_names[1:4])

# vec_to_save = np.array(['hi', 'bye', 'hello'],  dtype='str')

In [54]:
vec_to_save

['AU040320', 'Aacs', 'Aagab']

In [56]:
dset.create_dataset(name="test2", 
                    shape=(3, ),
                    dtype=h5py.string_dtype(encoding='utf-8'), 
                    data = vec_to_save)

<HDF5 dataset "test2": shape (3,), type "|O">

In [ ]:
dset.create_dataset(name="col_genenames", 
                    shape=gene_counts_anndata.var_names.shape,
                    dtype=h5py.string_dtype(encoding='utf-8'), 
                    data = gene_counts_anndata.var_names)


<HDF5 dataset "col_genenames": shape (4814,), type "|O">

In [58]:
dset.create_dataset(name="row_cellnames", 
                    shape=gene_counts_anndata.obs_names.shape,
                    dtype=h5py.string_dtype(encoding='utf-8'), 
                    data = gene_counts_anndata.obs_names)


<HDF5 dataset "row_cellnames": shape (253587,), type "|O">

In [88]:
(gene_counts.shape[0], )

(253587,)

In [8]:
gene_counts_anndata.obs_names
gene_counts_anndata.var_names

Index(['AI593442', 'AU040320', 'Aacs', 'Aagab', 'Aak1', 'Aamp', 'Aar2', 'Aars',
       'Aarsd1', 'Aasdhppt',
       ...
       'mt-Atp8', 'mt-Co2', 'mt-Co3', 'mt-Cytb', 'mt-Nd1', 'mt-Nd2', 'mt-Nd3',
       'mt-Nd4', 'mt-Nd4l', 'mt-Nd5'],
      dtype='str', length=4814)

In [ ]:
gene_counts = jin2025[1:4, normalized_genes_names].X

In [53]:
gene_counts

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 10972 stored elements and shape (3, 4814)>

In [49]:
jin2025[:, normalized_genes_names].obs_names

Index(['ACAAGCTAGCACGTTCACCAAGAACT-1:SCP038:10pert',
       'AGTGCAAGTTCGATTAGAAGCTTAAC-1:SCP038:10pert',
       'CGCAGGTAGACCAGTGTGATAACCCT-1:SCP038:10pert',
       'CTCCGGATCGATACTTACCACGTTAG-1:SCP038:10pert',
       'AACGCCCGTGATTACCCGCACAGTAT-1:SCP039:10pert',
       'CTAATTATCCGTCGCAACACTTGGAG-1:SCP039:10pert',
       'CTCACACAGTGAGAGACTGACACACA-1:SCP039:10pert',
       'GTCCCTTGTGAGATATCTCAGCCATC-1:SCP039:10pert',
       'ATTCGTAGTGATCCTGACAAGCTTGT-1:SCP040:10pert',
       'GCGCGATAGAACCCAGTCCAGCCAGA-1:SCP040:10pert',
       ...
       'TGAGAGAAGGATCGCAATGACGTTCA-1:SCP076:48pert',
       'TGAGCGCAGGGTGTCAGAAACCATGC-1:SCP076:48pert',
       'TGAGGCATCTGATAGAAGGTATGTAG-1:SCP076:48pert',
       'TGAGTCGAGTGTCGCTTCGTAAGCCA-1:SCP076:48pert',
       'TGCGTGGCACCATGCCTCGCGTTATA-1:SCP076:48pert',
       'TGGCTAGAGGAGTCTGACAACCGAAG-1:SCP076:48pert',
       'TGTAACCGTGGGTAACTGTGAACCTC-1:SCP076:48pert',
       'TGTAGCTGTATACTGCGAGAGTCGAC-1:SCP076:48pert',
       'TGTAGTAAGGCCAGGTGCGAATAGAT-

In [57]:
jin2025.n_obs

253587

In [56]:
len(normalized_genes_names)


4814

In [ ]:
jin2025.obs

,scDblFinder.class,scDblFinder.score,scDblFinder.weighted,scDblFinder.cxds_score,sample_label,sample_id,species,source,tissue,sex,...,predicted_supertype,predicted_supertype_probability,predicted_cluster,predicted_cluster_probability,cell_type,neuron_type,neighborhood,region_level1,region_level2,predicted_group
ACAAGCTAGCACGTTCACCAAGAACT-1:SCP038:10pert,singlet,0.000005,0.052418,0.129333,7L,SL054,mouse,mouse7,whole brain,F,...,0001 CLA-EPd-CTX Car3 Glut_1,0.99,0003 CLA-EPd-CTX Car3 Glut_1,0.88,Neuron,Glut,Pallium-Glut,Cortex,Isocortex,001 L5-6 IT Glut
AGTGCAAGTTCGATTAGAAGCTTAAC-1:SCP038:10pert,singlet,0.000021,0.048683,0.166063,4L,SL051,mouse,mouse4,whole brain,F,...,0001 CLA-EPd-CTX Car3 Glut_1,1.00,0002 CLA-EPd-CTX Car3 Glut_1,0.91,Neuron,Glut,Pallium-Glut,Cortex,Isocortex,001 L5-6 IT Glut
CGCAGGTAGACCAGTGTGATAACCCT-1:SCP038:10pert,singlet,0.000010,0.055799,0.110633,4L,SL051,mouse,mouse4,whole brain,F,...,0001 CLA-EPd-CTX Car3 Glut_1,1.00,0001 CLA-EPd-CTX Car3 Glut_1,0.99,Neuron,Glut,Pallium-Glut,Cortex,Isocortex,001 L5-6 IT Glut
CTCCGGATCGATACTTACCACGTTAG-1:SCP038:10pert,singlet,0.000208,0.201630,0.082096,6L,SL053,mouse,mouse6,whole brain,F,...,0002 CLA-EPd-CTX Car3 Glut_2,0.66,0005 CLA-EPd-CTX Car3 Glut_2,0.66,Neuron,Glut,Pallium-Glut,Cortex,Isocortex,001 L5-6 IT Glut
AACGCCCGTGATTACCCGCACAGTAT-1:SCP039:10pert,singlet,0.006852,0.141636,0.204695,7L,SL054,mouse,mouse7,whole brain,F,...,0001 CLA-EPd-CTX Car3 Glut_1,1.00,0001 CLA-EPd-CTX Car3 Glut_1,1.00,Neuron,Glut,Pallium-Glut,Cortex,Isocortex,001 L5-6 IT Glut
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TGGCTAGAGGAGTCTGACAACCGAAG-1:SCP076:48pert,singlet,0.000009,0.117418,0.138332,66L,SL183,mouse,mouse66,whole brain,F,...,0675 PF Fzd5 Glut_2,1.00,2724 PF Fzd5 Glut_2,0.76,Neuron,Glut,TH-EPI-Glut,Interbrain,TH,145 MH-LH TH Glut
TGTAACCGTGGGTAACTGTGAACCTC-1:SCP076:48pert,singlet,0.005369,0.159382,0.231852,71R,SL214,mouse,mouse71,whole brain,M,...,0675 PF Fzd5 Glut_2,1.00,2720 PF Fzd5 Glut_2,1.00,Neuron,Glut,TH-EPI-Glut,Interbrain,TH,145 MH-LH TH Glut
TGTAGCTGTATACTGCGAGAGTCGAC-1:SCP076:48pert,singlet,0.000010,0.130869,0.159221,73R,SL216,mouse,mouse73,whole brain,F,...,0675 PF Fzd5 Glut_2,1.00,2723 PF Fzd5 Glut_2,1.00,Neuron,Glut,TH-EPI-Glut,Interbrain,TH,145 MH-LH TH Glut
TGTAGTAAGGCCAGGTGCGAATAGAT-1:SCP076:48pert,singlet,0.000007,0.097777,0.145674,68L,SL185,mouse,mouse68,whole brain,M,...,0675 PF Fzd5 Glut_2,1.00,2725 PF Fzd5 Glut_2,0.81,Neuron,Glut,TH-EPI-Glut,Interbrain,TH,145 MH-LH TH Glut
